# AI-Accelerated QEC for Neutral-Atom Logical Qubits

This interactive companion to Infleqtion’s [AI-accelerated QEC post](https://infleqtion.com/ai-accelerated-qec/) teaches two neural-predecoding workflows side by side: a loss-free model trained on Stim circuit-level Pauli data, then a loss-aware model trained on Clifft leakage/loss trajectories. Change the noise probabilities, number of shots, and execution mode, regenerate the metrics, and inspect each workflow's logical-error-rate comparison.

> **Tutorial simulation.** Both stages use a distance-9 surface-code memory circuit. Stim covers the ordinary Pauli-noise case; Clifft extends it with five-level leakage/loss trajectories. Neither is a hardware-calibrated prediction.

## Why decoding matters

QEC is both a quantum and a classical-computing problem. Every correction round creates syndrome data that must be processed fast enough to keep pace with the hardware. This tutorial starts with a learned local pass that simplifies Pauli-noise syndrome information, then asks what must change when neutral-atom leakage and loss are present.

For neutral atoms, this is especially relevant as measurement and control loops become faster: useful logical qubits require accuracy, throughput, and low decoding latency together.

## Neutral atoms are qubits in theory, but qudits in practice

Neutral-atom hardware can occupy levels outside the computational subspace. In this simplified model, `|0L⟩` and `|1L⟩` leak out of the computational space but are still read as 0-like and 1-like outcomes. The decoder therefore sees leakage indirectly through a syndrome stream shaped by both Pauli-type noise and out-of-subspace population.

Run the next cell to explore that binary readout mapping.

In [ ]:
def map_readout_state(level: int) -> int:
    """Map computational and leakage levels to a binary readout."""
    zero_like = {0, 2}  # |0⟩ and |0L⟩
    one_like = {1, 3}   # |1⟩ and |1L⟩
    if level in zero_like:
        return 0
    if level in one_like:
        return 1
    raise ValueError(f"Unexpected state label: {level}")

for level, label in enumerate(("|0⟩", "|1⟩", "|0L⟩", "|1L⟩")):
    print(f"{label:4} → observed binary outcome {map_readout_state(level)}")

## Experiment setup

This tutorial is run from the checked-out repository, including `third_party/ising-decoding`. Stage 1 uses Stim's `FlipSimulator` to train a four-channel `PreDecoderModelMemory_v1` on circuit-level Pauli noise. Stage 2 uses Clifft's five-level leakage/loss simulator to train the loss-aware five-channel `PreDecoderModelMemory_v2`. Both models produce correction/residual outputs that are completed by PyMatching. Create a project-local GPU environment before opening the notebook:

```bash
cd ~/Ieee_qec_26_tutorial
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install torch numpy matplotlib pymatching clifft jupyterlab ipykernel stim
python -m ipykernel install --user --name ieee-qec --display-name "Python (IEEE QEC)"
```

Launch Jupyter from this same directory with `.venv` activated, then select the Python (IEEE QEC) kernel. Confirm `torch.cuda.is_available()` is `True` before running the pipeline. **Stage 1:** the model sees X/Z syndromes and their geometry masks. **Stage 2:** Clifft additionally simulates leaked and lost levels, removes later interactions with lost atoms, and emits a measurement-loss herald sidecar; the model sees the same four channels plus that herald channel.

In [ ]:
import re
import os
import subprocess
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

mpl.rcParams.update({"font.size": 12, "axes.titlesize": 13, "axes.labelsize": 12})
%config InlineBackend.figure_format = 'retina'

In [ ]:
repo_dir = Path.cwd().resolve()
ising_dir = repo_dir / "third_party/ising-decoding"
scripts_dir = ising_dir / "code/scripts"
data_dir = repo_dir / "data"
output_dir = repo_dir / "outputs"
if not scripts_dir.is_dir():
    raise RuntimeError("Missing Ising-Decoding submodule. Run `git submodule update --init --recursive`.")
print(f"Using Ising-Decoding checkout: {ising_dir}")
print("Tutorial stages: Stim four-channel predecoder; Clifft five-channel loss-aware predecoder")

## Run controls

The notebook trains both models live on CUDA. First, Stim samples ordinary circuit-level Pauli noise and trains a four-channel neural predecoder. Second, Clifft samples leakage/loss trajectories and trains a five-channel loss-aware neural predecoder. Each stage is evaluated against PyMatching on held-out data generated by that same simulator. Select smoke only to test the wiring quickly.

In [ ]:
# Run these cells in order on a CUDA kernel.
EXECUTION_MODE = "gpu"  # choose "smoke" or "gpu"
RUN_SEED = None  # None creates a fresh random training/held-out split; set an integer to reproduce it.

if EXECUTION_MODE == "smoke":
    SHOTS, EPOCHS, BATCH_SIZE = 4_096, 3, 256
elif EXECUTION_MODE == "gpu":
    SHOTS, EPOCHS, BATCH_SIZE = 65_536, 20, 256
else:
    raise ValueError("EXECUTION_MODE must be 'smoke' or 'gpu'.")

PAULI_ERROR_PROBABILITY = 0.002
LEAKAGE_PROBABILITY = 0.001
LOSS_PROBABILITY = 0.004
LOSS_BOUNDARY_WEIGHT = 2.0  # PyMatching fallback for detector parities caused by physical loss

print({"mode": EXECUTION_MODE, "shots": SHOTS, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "seed": RUN_SEED, "pauli_p": PAULI_ERROR_PROBABILITY, "leakage_p": LEAKAGE_PROBABILITY, "loss_p": LOSS_PROBABILITY})

In [ ]:
# Stage 1: Stim Pauli trajectories → four-channel neural predecoder → PyMatching.
# Stage 2: Clifft leakage/loss trajectories → five-channel neural predecoder → PyMatching.
runner_python = repo_dir / ".venv/bin/python"
if not runner_python.is_file():
    raise RuntimeError(f"Create the project environment first: python3 -m venv {repo_dir / '.venv'}")

script_env = os.environ | {"PYTHONPATH": str(ising_dir / "code")}

def run(script, *args, capture=False):
    command = [str(runner_python), str(repo_dir / script), *map(str, args)]
    print("$", " ".join(command))
    completed = subprocess.run(
        command, cwd=repo_dir, env=script_env, check=True, text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None,
    )
    if capture:
        print(completed.stdout, end="")
    return completed.stdout

run_seed = int(np.random.SeedSequence().generate_state(1)[0]) if RUN_SEED is None else RUN_SEED
test_shots = max(512, SHOTS // 8)
stim_stdout = run(
    "stim_neural_predecoder.py",
    "--distance", 9, "--rounds", 9,
    "--train-shots", SHOTS, "--test-shots", test_shots,
    "--pauli-p", PAULI_ERROR_PROBABILITY,
    "--epochs", EPOCHS, "--batch-size", BATCH_SIZE, "--device", "cuda",
    "--seed", run_seed, capture=True,
)
stim_metrics = dict(re.findall(r"^(PyMatching baseline logical error rate|Stim-trained local predecoder \+ PyMatching logical error rate): ([0-9.eE+-]+)%?$", stim_stdout, flags=re.MULTILINE))
stim_result = {
    "dataset": f"fresh Stim d=9 Pauli-only sample ({SHOTS:,} training shots; seed={run_seed})",
    "predecoder": "newly trained four-channel local CNN",
    "input_channels": "X syndrome, Z syndrome, X geometry, Z geometry",
    "baseline_ler": float(stim_metrics["PyMatching baseline logical error rate"]) / 100,
    "predecoded_ler": float(stim_metrics["Stim-trained local predecoder + PyMatching logical error rate"]) / 100,
}

clifft_stdout = run(
    "clifft_ising_loss_aware_predecoder.py",
    "--distance", 9, "--rounds", 9,
    "--train-shots", SHOTS, "--test-shots", test_shots,
    "--pauli-p", PAULI_ERROR_PROBABILITY, "--leakage-p", LEAKAGE_PROBABILITY, "--loss-p", LOSS_PROBABILITY,
    "--epochs", EPOCHS, "--batch-size", BATCH_SIZE, "--device", "cuda",
    "--seed", run_seed, "--loss-boundary-weight", LOSS_BOUNDARY_WEIGHT, capture=True,
)
clifft_metrics = dict(re.findall(r"^(PyMatching baseline logical error rate|loss-aware local predecoder \+ PyMatching logical error rate|mean predicted local correction density): ([0-9.eE+-]+)%?$", clifft_stdout, flags=re.MULTILINE))
clifft_result = {
    "dataset": f"fresh Clifft d=9 leakage/loss sample ({SHOTS:,} training shots; seed={run_seed})",
    "predecoder": "newly trained five-channel, four-head local Ising-style predecoder",
    "downstream_decoder": "PyMatching on the Pauli graph with loss fallback boundaries",
    "input_channels": "X syndrome, Z syndrome, X geometry, Z geometry, loss herald",
    "local_correction_density": float(clifft_metrics["mean predicted local correction density"]),
    "baseline_ler": float(clifft_metrics["PyMatching baseline logical error rate"]) / 100,
    "predecoded_ler": float(clifft_metrics["loss-aware local predecoder + PyMatching logical error rate"]) / 100,
}
results = {"Stim Pauli-only": stim_result, "Clifft leakage/loss": clifft_result}

## Results: what the loss-herald channel changes

**Stage 1** is a conventional local neural predecoder: Stim's `FlipSimulator` generates circuit-level Pauli samples and exposes hidden Pauli-frame targets during training. The model receives four channels—X/Z syndromes plus the corresponding geometry masks—then passes its residual detector record to PyMatching.

**Stage 2** adds the neutral-atom complication: Clifft directly simulates leaked and lost states. The fifth channel is the final data-readout loss herald, broadcast across the space-time input. Its conservative local teacher proposes a frame correction only for unambiguous heralded-loss regions; ambiguous effects remain in the residual for PyMatching. Because physical loss can create a detector parity outside the Pauli-only DEM, this stage uses finite-cost detector-to-boundary fallback edges. Compare baseline versus predecoded results *within each simulator*, not across the two different noise models.

In [ ]:
# `results` is created by the live evaluation cell above; no stored data are loaded here.
if "results" not in globals():
    raise RuntimeError("Run the live pipeline cell first so both models are newly trained.")
for stage, result in results.items():
    print(f"\n{stage}")
    for name, value in result.items():
        print(f"{name}: {value}")

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
labels = ["PyMatching\nbaseline", "Neural predecoder +\nPyMatching"]
colors = ["tab:blue", "tab:green"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (stage, result) in zip(axes, results.items()):
    values = [result["baseline_ler"], result["predecoded_ler"]]
    bars = ax.bar(labels, values, color=colors)
    ax.bar_label(bars, labels=[f"{value:.3%}" for value in values], padding=3)
    ax.set_title(stage)
    ax.set_ylim(0, 1)
axes[0].set_ylabel("Held-out logical error rate")
fig.text(0.5, -0.05, "Each panel is a within-simulator comparison. Stage 2 adds Clifft leakage/loss data and a fifth loss-herald input channel.",
         ha="center", va="top", fontsize=10)
fig.tight_layout()
plt.show()

## From leakage to erasure

Leakage is not always an opaque error channel. Here, Clifft tracks two leaked levels and a physically lost level. Loss removes later multi-qubit interactions and emits a data-readout herald at the end of the block. The local predecoder receives that herald as its fifth channel while retaining the unmodified Ising syndrome layout.

This is a local correction/residual objective, not a claim that a local network resolves the global consequences of atom loss. PyMatching supplies the final global logical decision for both the baseline and locally reduced detector records.

## Why this matters for Sqale

The broader objective is a coherent hybrid QPU–GPU stack: strong physical qubits, scalable logical architectures, GPU-resident classical acceleration, and AI models that make control and decoding faster and more informed. For neutral atoms, Clifft provides an explicit noncomputational leakage/loss model, while the pipeline illustrates loss-aware local syndrome filtering followed by global matching.

Read the full [published post](https://infleqtion.com/ai-accelerated-qec/) for the neutral-atom context, platform roadmap, and source figures. This notebook is the executable companion for the loss-aware AI-decoding experiment.